In [56]:
from langchain_groq import ChatGroq
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv


In [57]:
load_dotenv()

True

In [58]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str
    
    

In [59]:
model=ChatGroq(
    model='llama-3.3-70b-versatile'
)

In [60]:
def generate_joke(state:JokeState):
    prompt=f'Generate a short and easy to understandable joke on this {state['topic']}.'
    responce=model.invoke(prompt)
    
    return {'joke':responce}

In [61]:
def generate_explanation(state:JokeState):
    prompt=f'Generate a simple easy to understandable explanation of the given joke {state['joke']}.'
    responce=model.invoke(prompt)
    
    return {'explanation':responce}

In [62]:
thread_id = '1'

graph = StateGraph(JokeState)
check_point = InMemorySaver()

config = {
    'configurable': {
        'thread_id': thread_id
    }
}

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

workflow = graph.compile(checkpointer=check_point)

In [ ]:
workflow.invoke({'topic':'Langgraph'},config=config)

<generator object Pregel.stream at 0x000002BC135B4E10>